# Data Understanding - Olist Dataset

In [1]:
!pwd

/content


In [3]:
#!/bin/bash
!curl -L -o /content/brazilian-ecommerce.zip\
  https://www.kaggle.com/api/v1/datasets/download/olistbr/brazilian-ecommerce

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 42.6M  100 42.6M    0     0  14.0M      0  0:00:03  0:00:03 --:--:-- 21.5M


In [4]:
!unzip /content/brazilian-ecommerce.zip -d /content/brazilian-ecommerce/

Archive:  /content/brazilian-ecommerce.zip
  inflating: /content/brazilian-ecommerce/olist_customers_dataset.csv  
  inflating: /content/brazilian-ecommerce/olist_geolocation_dataset.csv  
  inflating: /content/brazilian-ecommerce/olist_order_items_dataset.csv  
  inflating: /content/brazilian-ecommerce/olist_order_payments_dataset.csv  
  inflating: /content/brazilian-ecommerce/olist_order_reviews_dataset.csv  
  inflating: /content/brazilian-ecommerce/olist_orders_dataset.csv  
  inflating: /content/brazilian-ecommerce/olist_products_dataset.csv  
  inflating: /content/brazilian-ecommerce/olist_sellers_dataset.csv  
  inflating: /content/brazilian-ecommerce/product_category_name_translation.csv  


In [5]:
!ls -lrt -h /content/brazilian-ecommerce/

total 121M
-rw-r--r-- 1 root root 8.7M Oct  1  2021 olist_customers_dataset.csv
-rw-r--r-- 1 root root  59M Oct  1  2021 olist_geolocation_dataset.csv
-rw-r--r-- 1 root root  15M Oct  1  2021 olist_order_items_dataset.csv
-rw-r--r-- 1 root root 5.6M Oct  1  2021 olist_order_payments_dataset.csv
-rw-r--r-- 1 root root  17M Oct  1  2021 olist_orders_dataset.csv
-rw-r--r-- 1 root root  14M Oct  1  2021 olist_order_reviews_dataset.csv
-rw-r--r-- 1 root root 2.3M Oct  1  2021 olist_products_dataset.csv
-rw-r--r-- 1 root root 2.6K Oct  1  2021 product_category_name_translation.csv
-rw-r--r-- 1 root root 171K Oct  1  2021 olist_sellers_dataset.csv


In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('olist').getOrCreate()

In [8]:
customer_df = spark.read.format('csv').option('header', 'true').option('inferSchema', 'true').load('/content/brazilian-ecommerce/olist_customers_dataset.csv')

In [13]:
customer_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [17]:
customer_df.show()

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                    1151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                    8775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83...|345ecd01c38d18a90...|                   13056|            campinas|            SP|
|879864dab9bc30475...|4c93744516667ad3b...|                   89254|      jaragua do sul|            SC|
|fd826e7cf63160e53...|addec96d2e059c80c...|            

# Task
Automate the data loading, schema inspection, and preview for all CSV datasets located in the "/content/brazilian-ecommerce/" folder using a loop in PySpark.

## modify_cells

### Subtask:
Iterate through all CSV files in the Brazilian Ecommerce directory to automate loading, schema display, and data preview using PySpark.


## Summary:

### Q&A

**How can the data loading and inspection process for multiple datasets be streamlined in PySpark?**
The process is automated by using the Python `os` module to list all files in the target directory and filtering for those with a `.csv` extension. A `for` loop then iterates through these files, dynamically generating variable names and using `spark.read.csv` with `header=True` and `inferSchema=True` to load, print the schema, and display the first few rows of each dataset.

### Data Analysis Key Findings

*   **Dataset Variety:** The directory contains 9 distinct CSV files covering various aspects of the Brazilian Ecommerce ecosystem, including customers, geolocation, order items, order payments, order reviews, orders, products, sellers, and product category name translations.
*   **Schema Complexity:**
    *   **Orders & Items:** These tables contain complex temporal data (timestamps for purchase, approval, and delivery) and financial metrics (price and freight value).
    *   **Geolocation:** This is the largest spatial dataset, containing latitudes, longitudes, zip codes, cities, and states.
    *   **Reviews:** Includes both quantitative scores (1-5) and qualitative text data (review comments).
*   **Data Types:** PySpark successfully inferred data types across the files, identifying `StringType` for IDs and categories, `DoubleType` for financial values and coordinates, and `IntegerType` for counts and scores.

### Insights or Next Steps

*   **Standardize Timestamps:** Many datasets contain date strings that should be explicitly cast to `TimestampType` to facilitate time-series analysis or delivery performance calculations.
*   **Entity Resolution:** The next logical step is to perform join operations (e.g., linking `orders` to `order_items` and `customers`) to create a unified view for deeper business intelligence or machine learning modeling.


In [19]:
import os

directory_path = '/content/brazilian-ecommerce/'
csv_files = [f for f in os.listdir(directory_path) if f.endswith('.csv')]

for csv in csv_files:
    file_path = os.path.join(directory_path, csv)
    print(f"\n{'='*20} Loading: {csv} {'='*20}")

    # Load the dataset
    df = spark.read.format('csv') \
        .option('header', 'true') \
        .option('inferSchema', 'true') \
        .load(file_path)

    # Display schema and data
    df.printSchema()
    df.show(10, truncate=False)



==================== Loading: olist_customers_dataset.csv ====================
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

+--------------------------------+--------------------------------+------------------------+---------------------+--------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city        |customer_state|
+--------------------------------+--------------------------------+------------------------+---------------------+--------------+
|06b8999e2fba1a1fbc88172c00ba8bc7|861eff4711a542e4b93843c6dd7febb0|14409                   |franca               |SP            |
|18955e83d337fd6b2def6b18a428ac77|290c77bc529b7ac935b93aa66c333dc3|9790                    |sao bernardo do campo|SP            |
|4e7b3e00288586ebd08712